# 02 — Baselines: el piso a superar

**TP Final · Aprendizaje de Máquina I (CEIA-FIUBA)** · Jaime Pinzón (a2629)

## ¿Por qué un baseline, y por qué estos?

Un baseline responde la pregunta *"¿cuánto se puede lograr sin aprender casi nada?"*. Si un modelo complejo no supera con claridad este piso, su costo no se justifica — esa comparación es el eje del informe final.

- **`DummyRegressor(strategy="mean")`**: predice siempre la media del train. Es el predictor constante que minimiza el error cuadrático.
- **`DummyRegressor(strategy="median")`**: predice siempre la mediana. Es el predictor constante que minimiza el **MAE** — nuestra métrica principal — así que es el baseline "más difícil de vencer" entre los triviales.
- **Regresión lineal (OLS)**: la referencia paramétrica más simple que sí usa las features. Marca cuánta señal lineal hay en los datos.

**Nota metodológica:** la cátedra pide explícitamente NO usar regresión logística como baseline. Además de la indicación, hay una razón técnica: la regresión logística es un **clasificador** (modela probabilidades de clases discretas); nuestro problema es de **regresión** sobre días de internación, donde el análogo correcto es la regresión lineal.

Los tres baselines usan el mismo preprocesador que usarán todos los modelos (`build_preprocessor`), sin escalado: el Dummy ignora las features y OLS es invariante a transformaciones afines de las columnas.

In [1]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from src.config import DIR_PROCESSED, TARGET
from src.evaluacion import evaluar_y_registrar
from src.pipelines import build_preprocessor

X_train = pd.read_parquet(DIR_PROCESSED / "X_train.parquet")
X_test = pd.read_parquet(DIR_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(DIR_PROCESSED / "y_train.parquet")[TARGET]
y_test = pd.read_parquet(DIR_PROCESSED / "y_test.parquet")[TARGET]

print(f"train: {X_train.shape} | test: {X_test.shape}")
print(f"media del train: {y_train.mean():.3f} dias | mediana: {y_train.median():.1f} dias")

train: (53756, 8) | test: (20354, 8)
media del train: 4.087 dias | mediana: 3.0 dias


In [2]:
modelos = {
    "baseline_media": DummyRegressor(strategy="mean"),
    "baseline_mediana": DummyRegressor(strategy="median"),
    "regresion_lineal": LinearRegression(),
}
notas = {
    "baseline_media": "DummyRegressor(mean); predictor constante",
    "baseline_mediana": "DummyRegressor(median); minimiza MAE entre constantes",
    "regresion_lineal": "OLS sobre las 17 features del preprocesador comun",
}

resultados = {}
for nombre, estimador in modelos.items():
    pipe = Pipeline([
        ("preprocesador", build_preprocessor(scale=False)),
        ("modelo", estimador),
    ])
    pipe.fit(X_train, y_train)
    resultados[nombre] = evaluar_y_registrar(
        nombre, pipe, X_test, y_test, notas=notas[nombre]
    )

pd.DataFrame(resultados).T.round(4)

,mae,rmse,r2
baseline_media,2.2805,3.0009,-0.0107
baseline_mediana,2.2929,3.2952,-0.2187
regresion_lineal,1.8616,2.4666,0.3171


In [3]:
# El registro usa UPSERT por nombre: re-ejecutar este notebook NO duplica filas.
from src.evaluacion import RUTA_METRICAS, registrar

registrar("baseline_media", resultados["baseline_media"], notas=notas["baseline_media"])
tabla = pd.read_csv(RUTA_METRICAS)
assert tabla["modelo"].is_unique, "hay modelos duplicados en metricas.csv"
print(f"metricas.csv: {len(tabla)} filas, sin duplicados")
tabla.round(4)

metricas.csv: 3 filas, sin duplicados


,modelo,mae,rmse,r2,notas
0,baseline_media,2.2805,3.0009,-0.0107,DummyRegressor(mean); predictor constante
1,baseline_mediana,2.2929,3.2952,-0.2187,DummyRegressor(median); minimiza MAE entre con...
2,regresion_lineal,1.8616,2.4666,0.3171,OLS sobre las 17 features del preprocesador comun


## Lectura de negocio

- **El piso a superar es MAE ≈ 2,28 días** (baseline de media). La política ingenua "asumir que todo paciente se queda lo mismo que el paciente típico" se equivoca, en promedio, 2,28 días por paciente. Todo modelo de los notebooks 03–06 se juzga por cuántos días le recorta a ese piso.
- **Detalle honesto que muestra la tabla:** en teoría la mediana minimiza el MAE entre predictores constantes, pero acá la media (2,2805) le ganó por poco a la mediana (2,2929). La razón es metodológica y está buscada: los baselines se ajustan sobre el **train filtrado por IQR** (mediana 3 días, estadías largas removidas) pero se evalúan sobre el **test sin filtrar** (hasta 14 días). La mediana de 3 queda más lejos de las estadías largas del test que la media de 4,09. Es la primera evidencia visible de que evaluar sobre el test completo — como ocurriría en producción — penaliza los supuestos que solo valen en el train.
- La **regresión lineal** baja el MAE a 1,86 días (recorta 0,42 días, −18%) con R² = 0,32: hay señal lineal real en las 17 features, pero dos tercios de la varianza siguen sin explicarse — ese es el espacio que los modelos no lineales (KNN, SVR, árboles, ensambles) intentarán capturar.
- Contexto operativo: recortar 0,42 días de error medio, a escala de un hospital con miles de admisiones anuales, ya es capacidad de camas real; cada décima adicional que recorten los modelos siguientes se traduce directamente en mejor planificación de ocupación.

**Siguiente notebook (03):** KNN regressor — primer modelo real, con la justificación de por qué exige escalado.